# Experiment 14 — more profitable allocation

Based on [NB13](13-backtest-leading-candidates.ipynb) and [NB12](12-research-allocation-followup.ipynb). Implements the [reviewed plan](14-profitable-allocation-plan.md).

## Goal and method

Make the strategy more profitable through better capital placement and redistribution within caps. Compare source variance and regularised variance under original versus simultaneous allocation, plus regularised downside sizing under original allocation. Freeze C03's ranking (48% CAGR360, 32% Sortino45, 10% CAGR180, 10% Sortino30; existing clips), six positions, missing=True, two-day cadence, selection/gates/hysteresis and fees. Reuse all indicator warmup, causal NAV filling and actual-observation counts for partial-history regularisation. Test whether downside sizing retains profitable upward-moving vaults without unacceptable concentration in loss-free young histories.

Two independent calendars: **2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)** and **2023-03-01 → 2026-10-08 exclusive (through 7 October, UTC; final archive day partial)**. Every account starts with $150,000. Net dollars and CAGR are primary; drawdown, Sharpe, costs, exposure and vault/quarter contributions explain the economic trade-offs. Default presentation is the short 2026 period; long-history comparisons are explicitly labelled.

## Key new insights

2026: profit leader **A**. Best challenger **B** changes profit by **$-1,342**, CAGR by -1.31 percentage points, and drawdown by -0.19 percentage points versus A.

Long: profit leader **G**. Best challenger **G** changes profit by **$+51,687**, CAGR by +5.32 percentage points, and drawdown by -3.46 percentage points versus A.

## Summary of results

Five methods and both periods are completed, with A-versus-leading-challenger quote-freshness sensitivities. Tables below show the full profit/risk comparison. Detailed [findings](14-profitable-allocation-findings.md) retain attribution and limitations.

## Robustness of results

The reference timing trace confirmed later intraday NAV consumption; primary pricing is `as_of_observation`. Source indicators and scores still end at T−1. Every primary fill is checked against its recorded as-of NAV, allocations retain source caps, and position PnL reconciles to equity within $1. Quote-freshness sensitivities defer trading against NAV older than the source two-day cycle; actual execution NAV remains unverified where observations are sparse. These dates have already been searched and are not an independent holdout.

The inherited fee model is retained for comparability: partial trims can store a net-of-fee mid price and apply the redemption charge again. The recorded cost column is therefore the source estimate, rather than independently reconciled total fees. See the findings for this limitation.

The regularised estimator comparison includes support, NAV filling and fallback differences, so the historical gain cannot be attributed to shrinkage alone.


In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'profitable_allocation.py').exists():ROOT=ROOT/'scratchpad/hyperliquid-vov-2nd'
sys.path.insert(0,str(ROOT))
from profitable_allocation import initialise,run
from profitable_allocation_report import setup,report
from score_matrix_visualisation import compact_output


## Fix the reference and pricing convention — 2026-01-01 → 2026-10-05 exclusive

Load pinned inputs and all pre-decision history, reproduce saved A/D controls and inspect successful fills. Choose the shared observation-time convention before new replays. No operational permission field is filled by the NAV rule.

In [2]:
with compact_output('Pinned inputs, four legacy parity controls and timing trace; estimated 2–4 minutes'):
    data,evaluator,preflight=initialise()
setup(data,preflight)


Pinned inputs, four legacy parity controls and timing trace; estimated 2–4 minutes — complete

### Registered methods — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Ranking, six positions, missing=True, the two-day cadence and source constraints are fixed. C and D share the regularised risk estimate; G changes its individual and peer estimate to squared losses.

Arm,Method,Risk,Allocator
A,Source variance / original allocation,source,sequential
B,Source variance / simultaneous allocation,source,simultaneous
C,Regularised variance / original allocation,regular,sequential
D,Regularised variance / simultaneous allocation,regular,simultaneous
G,Regularised downside / original allocation,downside,sequential


### Price timing preflight — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Compare saved successful trade prices with the final daily observation and the latest NAV actually observed before execution. Archive write time is not used as a price-availability clock.

Period,Arm,Trades,Later_NAV_used,Mean_known_NAV_age_days
2026,A,569,0,0.2444642994735729
2026,D,598,0,0.27791967373885174


**Chosen convention:** `as_of_observation`. Across both reference periods, 24 successful trades consumed later intraday NAV marks. Detailed long-period exceptions are in the ignored audit ledger. Indicators still use completed T−1 buckets, which precede T; only execution and valuation queries receive the shared observation-time adapter when the trace confirms a leak.

## Run five allocation methods over both periods

Ten required primary results: full 2026 and full available history. Preserve source trade generation while separating risk estimates from allocation. Then compare A and each period's leading challenger when trading requires a NAV observation within the previous two-day source cycle.

In [3]:
with compact_output('Five methods × two periods plus sparse-price sensitivities; estimated 8–20 minutes'):
    results,runs,attribution,absolute_weights,sensitivity=run(data,evaluator)


Five methods × two periods plus sparse-price sensitivities; estimated 8–20 minutes — complete

## Profit, attribution and robustness — 2026-01-01 → 2026-10-05 exclusive

Default charts and tables use fresh-start 2026. Explicitly labelled long-period results show previously searched calendar years and carried capital; those accounts end three days later than the fresh 2026 comparison.

In [4]:
findings=report(data,results,runs,attribution,absolute_weights,sensitivity)


### Key economic results — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Each account starts with $150,000. Net profit includes the source performance-fee accounting and estimated redemption allowance. Sorted by dollars earned; drawdown and Sharpe show the accompanying risk.

Arm,Method,Final equity (USD),Net profit (USD),CAGR,Sharpe,Maximum drawdown
A,Source variance / original allocation,"$205,430","$55,430",51.77%,2.30,-6.35%
B,Source variance / simultaneous allocation,"$204,089","$54,089",50.46%,2.25,-6.54%
C,Regularised variance / original allocation,"$201,111","$51,111",47.56%,2.12,-6.93%
G,Regularised downside / original allocation,"$200,819","$50,819",47.28%,2.11,-7.72%
D,Regularised variance / simultaneous allocation,"$200,120","$50,120",46.60%,2.09,-7.05%


### Deployment, concentration and modelled costs — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Cash and exposure are fractions of whole account equity, including early undeployed days. Turnover and modelled redemption costs help explain differences in net profit.

Arm,Mean cash weight,Mean largest vault weight,Largest vault weight,Trades,Gross trade turnover (USD),Modelled redemption costs (USD)
A,10.18%,30.95%,32.91%,569,"$7,138,928","$13,511"
B,9.88%,30.97%,32.92%,579,"$7,210,443","$13,394"
C,10.08%,29.82%,32.80%,592,"$6,888,230","$13,033"
G,10.03%,29.23%,32.83%,593,"$6,708,922","$12,637"
D,9.67%,30.35%,32.85%,598,"$7,001,648","$12,949"


### Risk support and fallback counts — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

These are unique vault/decision rows where sizing was requested, not counts of trades or independent observations. Young histories remain eligible; flat or loss-free windows are labelled separately. Funded shares weight each sizing request by its final target dollars, not by realised future profit.

...

### NAV intervals behind the strongest updates — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Top three position contributions for A and the leading challenger. Actual marks can span a week; BTC/ETH context uses those same mark dates. The curator flags identify excluded or quarantined vaults, without inferring that a large gain is a data error.

Arm,Update,Vault,Update profit (USD),Interval_days,NAV_return,Excluded,Quarantined,BTCUSDT return,ETHUSDT return
A,2026-06-26 00:00:00,Realist Capital,"$7,140",2.04,+14.06%,False,False,-4.69%,-5.96%
A,2026-06-26 00:00:00,Goon Edging,"$4,626",2.06,+13.86%,False,False,-4.69%,-5.96%
A,2026-06-26 00:00:00,Octavious Maximus,$266,2.05,+4.13%,False,False,-4.69%,-5.96%
B,2026-06-26 00:00:00,Realist Capital,"$7,130",2.04,+14.06%,False,False,-4.69%,-5.96%
B,2026-06-26 00:00:00,Goon Edging,"$4,624",2.06,+13.86%,False,False,-4.69%,-5.96%
B,2026-06-26 00:00:00,Octavious Maximus,$266,2.05,+4.13%,False,False,-4.69%,-5.96%


### NAV intervals behind the strongest updates — 2023-03-01 → 2026-10-08 exclusive (through 7 October, UTC; final archive day partial)

Top three position contributions for A and the leading challenger. Actual marks can span a week; BTC/ETH context uses those same mark dates. The curator flags identify excluded or quarantined vaults, without inferring that a large gain is a data error.

Arm,Update,Vault,Update profit (USD),Interval_days,NAV_return,Excluded,Quarantined,BTCUSDT return,ETHUSDT return
A,2025-08-14 00:00:00,naze oppa degen nation,"$4,406",7.00,+8.70%,False,False,+7.23%,+28.94%
A,2025-08-14 00:00:00,DailyTradeAI Low risk,"$3,612",7.03,+7.18%,False,False,+7.23%,+28.94%
A,2025-08-14 00:00:00,cDOGE,"$3,026",6.97,+15.85%,False,False,+7.23%,+28.94%
G,2024-11-13 00:00:00,Road to 10 Million,"$15,071",7.00,+30.26%,False,False,+19.59%,+17.09%
G,2024-11-13 00:00:00,FFLGN,"$4,326",6.07,+41.69%,False,False,+19.59%,+17.09%
G,2024-11-13 00:00:00,Delta_01,"$3,738",6.09,+7.26%,False,False,+19.59%,+17.09%


### Economic verdict — 2026-01-01 → 2026-10-05 exclusive (through 4 October, UTC)

Allocator: 2026 $-1,342; full history $+1,414. The profit hypothesis is supported over full history and rejected for the fresh 2026 comparison.

Regularisation: 2026 $-4,319; full history $+48,335. The profit hypothesis is supported over full history and rejected for the fresh 2026 comparison.

Downside vs regular: 2026 $-292; full history $+3,352. The profit hypothesis is supported over full history and rejected for the fresh 2026 comparison.

Redistribution alone contributes little: most of the full-history improvement is already present with regularisation under the original allocator. Downside sizing adds only about $3,352 over that regularised control and has worse drawdown in both primary periods. Keep A as the 2026 reference; G is a historical profit challenger, not an established replacement.

[Detailed findings](14-profitable-allocation-findings.md) retain both periods, sensitivities and execution limitations.